# QIIME 2 Moving Pictures Tutorial 🎥

本 notebook 依照官方 [Moving Pictures tutorial](https://moving-pictures-tutorial.readthedocs.io/en/2026.7/) 改寫，
使用 `rachis-qiime2-2026.7` conda 環境所註冊的 Jupyter kernel 執行。

所有 QIIME 2 指令皆以 `%%bash` cell 執行 shell 指令。執行前請確認：

- 已啟用 QIIME 2 amplicon distribution 的 conda 環境（此 notebook 使用的 kernel 即為該環境）
- 目前工作目錄有足夠空間存放下載的資料與分析結果

## 目錄
1. Sample metadata
2. Obtaining and importing data
3. Demultiplexing sequences
4. Sequence quality control and feature table construction (DADA2)
5. FeatureTable and FeatureData summaries
6. Generate a tree for phylogenetic diversity analyses
7. Alpha and beta diversity analysis
8. Alpha rarefaction plotting
9. Taxonomic analysis
10. Differential abundance testing with ANCOM-BC


In [2]:
import os, sys

# %%bash 開的是全新的 subshell，PATH 不會自動包含目前 kernel（qiime2 conda env）的 bin 目錄，
# 所以直接呼叫 `qiime` 會出現 "command not found"（exit code 127）。
# 這裡把目前 kernel 所在環境的 bin 目錄加進 PATH，之後所有 %%bash cell 都能繼承到。
env_bin = os.path.dirname(sys.executable)
if env_bin not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = env_bin + os.pathsep + os.environ["PATH"]

print("kernel python:", sys.executable)
print("PATH now starts with:", os.environ["PATH"].split(os.pathsep)[0])


kernel python: /work/c00cjz00/Miniconda/envs/rachis-qiime2-2026.7/bin/python
PATH now starts with: /work/c00cjz00/Miniconda/envs/rachis-qiime2-2026.7/bin


In [3]:
%%bash
# 確認 QIIME 2 已正確安裝
qiime info


QIIME is caching your current deployment for improved performance. This may take a few moments and should only happen once per deployment.


System versions
Python version: 3.12.13
Parsl version: 2026.2.23
rachis release: 2026.7
rachis version: 2026.7.0
q2cli version: 2026.7.0

Installed plugins
alignment: 2026.7.0
boots: 2026.7.0
composition: 2026.7.0
cutadapt: 2026.7.0
dada2: 2026.7.0
deblur: 2026.7.0
demux: 2026.7.0
diversity: 2026.7.0
diversity-lib: 2026.7.0
emperor: 2026.7.0
feature-classifier: 2026.7.0
feature-table: 2026.7.0
fondue: 2026.7.0
fragment-insertion: 2026.7.0
kmerizer: 2026.7.0
longitudinal: 2026.7.0
metadata: 2026.7.0
phylogeny: 2026.7.0
quality-control: 2026.7.0
quality-filter: 2026.7.0
rescript: 2026.7.0
sample-classifier: 2026.7.0
stats: 2026.7.0
taxa: 2026.7.0
types: 2026.7.0
vizard: 2026.7.0
vsearch: 2026.7.0

Application config directory
/home/c00cjz00/.config/q2cli

Config
Config Source: vendored config dict

Getting help
To find help and learning resources, visit https://qiime2.org.
To get help with configuring and/or understanding QIIME 2 parallelization, visit https://use.qiime2.org/en/stable/re

In [4]:
%%bash
mkdir -p qiime2-moving-pictures-tutorial
cd qiime2-moving-pictures-tutorial
pwd


/work/c00cjz00/notebook/qiime2-moving-pictures-tutorial


## 1. Sample metadata

下載樣本 metadata，並用 `qiime metadata tabulate` 產生可瀏覽的視覺化檔案。

> **Warning：** 請勿在 metadata 中放入機密資訊（如個資），因為 QIIME 2 會把 metadata 記錄在 provenance 中並隨結果保留。


In [5]:
%%bash
cd qiime2-moving-pictures-tutorial
wget -O 'sample-metadata.tsv' \
  'https://moving-pictures-tutorial.readthedocs.io/en/2026.7/data/moving-pictures/sample-metadata.tsv'


--2026-09-04 16:19:25--  https://moving-pictures-tutorial.readthedocs.io/en/2026.7/data/moving-pictures/sample-metadata.tsv
Resolving moving-pictures-tutorial.readthedocs.io (moving-pictures-tutorial.readthedocs.io)... 104.16.254.120, 104.16.253.120, 2606:4700::6810:fe78, ...
Connecting to moving-pictures-tutorial.readthedocs.io (moving-pictures-tutorial.readthedocs.io)|104.16.254.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2096 (2.0K) [text/tab-separated-values]
Saving to: ‘sample-metadata.tsv’

     0K ..                                                    100% 47.1M=0s

2026-09-04 16:19:25 (47.1 MB/s) - ‘sample-metadata.tsv’ saved [2096/2096]



In [6]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime metadata tabulate \
  --m-input-file sample-metadata.tsv \
  --o-visualization sample-metadata-viz.qzv


Saved Visualization to: sample-metadata-viz.qzv


## 2. Obtaining and importing data

下載多重序列（尚未依樣本拆分）並匯入為 QIIME 2 artifact（`EMPSingleEndSequences`）。


In [7]:
%%bash
cd qiime2-moving-pictures-tutorial
wget -O 'emp-single-end-sequences.zip' \
  'https://moving-pictures-tutorial.readthedocs.io/en/2026.7/data/moving-pictures/emp-single-end-sequences.zip'

unzip -d emp-single-end-sequences emp-single-end-sequences.zip


--2026-09-04 16:20:02--  https://moving-pictures-tutorial.readthedocs.io/en/2026.7/data/moving-pictures/emp-single-end-sequences.zip
Resolving moving-pictures-tutorial.readthedocs.io (moving-pictures-tutorial.readthedocs.io)... 104.16.254.120, 104.16.253.120, 2606:4700::6810:fe78, ...
Connecting to moving-pictures-tutorial.readthedocs.io (moving-pictures-tutorial.readthedocs.io)|104.16.254.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29096655 (28M) [application/zip]
Saving to: ‘emp-single-end-sequences.zip’

     0K .......... .......... .......... .......... ..........  0%  721K 39s
    50K .......... .......... .......... .......... ..........  0% 1.70M 28s
   100K .......... .......... .......... .......... ..........  0% 1.01M 28s
   150K .......... .......... .......... .......... ..........  0% 2.50M 23s
   200K .......... .......... .......... .......... ..........  0% 27.6M 19s
   250K .......... .......... .......... .......... ..........  1% 6

Archive:  emp-single-end-sequences.zip
  inflating: emp-single-end-sequences/sequences.fastq.gz  
  inflating: emp-single-end-sequences/barcodes.fastq.gz  


In [8]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime tools import \
  --type 'EMPSingleEndSequences' \
  --input-path emp-single-end-sequences \
  --output-path emp-single-end-sequences.qza


Imported emp-single-end-sequences as EMPSingleEndDirFmt to emp-single-end-sequences.qza


## 3. Demultiplexing sequences

根據 metadata 裡的 barcode 欄位（`barcode-sequence`）將序列拆分到各樣本。


In [9]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime demux emp-single \
  --i-seqs emp-single-end-sequences.qza \
  --m-barcodes-file sample-metadata.tsv \
  --m-barcodes-column barcode-sequence \
  --o-per-sample-sequences demux.qza \
  --o-error-correction-details demux-details.qza


Saved SampleData[SequencesWithQuality] to: demux.qza
Saved ErrorCorrectionDetails to: demux-details.qza


In [10]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime demux summarize \
  --i-data demux.qza \
  --o-visualization demux.qzv


Saved Visualization to: demux.qzv


> **提示：** 請用 [QIIME 2 View](https://view.qiime2.org) 開啟 `demux.qzv`，查看 Interactive Quality Plot，
> 用來決定下一步 DADA2 的 `--p-trim-left` 與 `--p-trunc-len` 參數。


## 4. Sequence quality control and feature table construction — DADA2

依據 `demux.qzv` 品質圖，這裡選擇不裁切開頭（`--p-trim-left 0`），並在第 120 個鹼基處截斷（`--p-trunc-len 120`）。

> 這一步可能需要跑到 10 分鐘，是整份教學中最慢的步驟。


In [11]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime dada2 denoise-single \
  --i-demultiplexed-seqs demux.qza \
  --p-trim-left 0 \
  --p-trunc-len 120 \
  --o-representative-sequences rep-seqs.qza \
  --o-table table.qza \
  --o-denoising-stats denoising-stats.qza \
  --o-base-transition-stats base-transition-stats.qza


Saved FeatureTable[Frequency] to: table.qza
Saved FeatureData[Sequence] to: rep-seqs.qza
Saved SampleData[DADA2Stats] to: denoising-stats.qza
Saved DADA2BaseTransitionStats to: base-transition-stats.qza


In [12]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime metadata tabulate \
  --m-input-file denoising-stats.qza \
  --o-visualization denoising-stats.qzv


Saved Visualization to: denoising-stats.qzv


### （選用）Option 2: Deblur

如果想改用 Deblur 而非 DADA2，可以執行下面的 cell（先將上面 DADA2 的 cell 略過或不執行）。
產生的 `rep-seqs-deblur.qza` / `table-deblur.qza` 最後需 `mv` 成 `rep-seqs.qza` / `table.qza` 才能接續後續步驟。


In [13]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime quality-filter q-score \
  --i-demux demux.qza \
  --o-filtered-sequences demux-filtered.qza \
  --o-filter-stats demux-filter-stats.qza


Saved SampleData[SequencesWithQuality] to: demux-filtered.qza
Saved QualityFilterStats to: demux-filter-stats.qza


In [14]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime deblur denoise-16S \
  --i-demultiplexed-seqs demux-filtered.qza \
  --p-trim-length 120 \
  --p-sample-stats \
  --o-representative-sequences rep-seqs-deblur.qza \
  --o-table table-deblur.qza \
  --o-stats deblur-stats.qza


Saved FeatureTable[Frequency] to: table-deblur.qza
Saved FeatureData[Sequence] to: rep-seqs-deblur.qza
Saved DeblurStats to: deblur-stats.qza


In [15]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime metadata tabulate \
  --m-input-file demux-filter-stats.qza \
  --o-visualization demux-filter-stats.qzv
qiime deblur visualize-stats \
  --i-deblur-stats deblur-stats.qza \
  --o-visualization deblur-stats.qzv


Saved Visualization to: demux-filter-stats.qzv
Saved Visualization to: deblur-stats.qzv


In [16]:
%%bash
# 若要改用 Deblur 的結果接續後續分析，取消下面兩行的註解並執行
cd qiime2-moving-pictures-tutorial
# mv rep-seqs-deblur.qza rep-seqs.qza
# mv table-deblur.qza table.qza


## 5. FeatureTable and FeatureData summaries

檢視每個樣本 / 每個 feature 的序列數量分布，以及 feature ID 對應序列（可直接 BLAST）。


In [17]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime feature-table summarize \
  --i-table table.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-summary table.qzv \
  --o-feature-frequencies feature-frequencies.qza \
  --o-sample-frequencies sample-frequencies.qza
qiime feature-table tabulate-seqs \
  --i-data rep-seqs.qza \
  --o-visualization rep-seqs.qzv


Saved ImmutableMetadata to: feature-frequencies.qza
Saved ImmutableMetadata to: sample-frequencies.qza
Saved Visualization to: table.qzv
Saved Visualization to: rep-seqs.qzv


## 6. Generate a tree for phylogenetic diversity analyses

用 `align-to-tree-mafft-fasttree` pipeline：MAFFT 多序列比對 → 過濾高變異位點 → FastTree 建樹 → midpoint rooting。


In [18]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime phylogeny align-to-tree-mafft-fasttree \
  --i-sequences rep-seqs.qza \
  --o-alignment aligned-rep-seqs.qza \
  --o-masked-alignment masked-aligned-rep-seqs.qza \
  --o-tree unrooted-tree.qza \
  --o-rooted-tree rooted-tree.qza


Saved FeatureData[AlignedSequence] to: aligned-rep-seqs.qza
Saved FeatureData[AlignedSequence] to: masked-aligned-rep-seqs.qza
Saved Phylogeny[Unrooted] to: unrooted-tree.qza
Saved Phylogeny[Rooted] to: rooted-tree.qza


## 7. Alpha and beta diversity analysis

用 `core-metrics-phylogenetic` 一次計算多種 alpha / beta diversity 指標，並產生 Emperor PCoA 圖。

`--p-sampling-depth` 需要參考上面 `table.qzv` 的 Interactive Sample Detail 分頁來決定；
這裡沿用官方教學建議值 `1103`（依 L3S313 樣本的序列數決定，會排除序列數過低的右手掌樣本）。

> 若你改用 Deblur 產生的 feature table，序列數分布可能不同，請重新評估這個值。


In [19]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime diversity core-metrics-phylogenetic \
  --i-phylogeny rooted-tree.qza \
  --i-table table.qza \
  --p-sampling-depth 1103 \
  --m-metadata-file sample-metadata.tsv \
  --output-dir diversity-core-metrics-phylogenetic


Saved FeatureTable[Frequency] to: diversity-core-metrics-phylogenetic/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: diversity-core-metrics-phylogenetic/faith_pd_vector.qza
Saved SampleData[AlphaDiversity] to: diversity-core-metrics-phylogenetic/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: diversity-core-metrics-phylogenetic/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: diversity-core-metrics-phylogenetic/evenness_vector.qza
Saved DistanceMatrix to: diversity-core-metrics-phylogenetic/unweighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: diversity-core-metrics-phylogenetic/weighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: diversity-core-metrics-phylogenetic/jaccard_distance_matrix.qza
Saved DistanceMatrix to: diversity-core-metrics-phylogenetic/bray_curtis_distance_matrix.qza
Saved PCoAResults to: diversity-core-metrics-phylogenetic/unweighted_unifrac_pcoa_results.qza
Saved PCoAResults to: diversity-core-metrics-phylogenetic

### Alpha diversity：群組間顯著性檢定（Faith's PD、Evenness）

In [20]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime diversity alpha-group-significance \
  --i-alpha-diversity diversity-core-metrics-phylogenetic/faith_pd_vector.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization faith-pd-group-significance.qzv
qiime diversity alpha-group-significance \
  --i-alpha-diversity diversity-core-metrics-phylogenetic/evenness_vector.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization evenness-group-significance.qzv


Saved Visualization to: faith-pd-group-significance.qzv
Saved Visualization to: evenness-group-significance.qzv


> **Question：** 哪些 categorical metadata 欄位與群落豐富度（richness）差異最相關？是否具統計顯著性？
> 哪些欄位與均勻度（evenness）差異最相關？


### Beta diversity：PERMANOVA 群組間顯著性檢定（unweighted UniFrac）

In [21]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime diversity beta-group-significance \
  --i-distance-matrix diversity-core-metrics-phylogenetic/unweighted_unifrac_distance_matrix.qza \
  --m-metadata-file sample-metadata.tsv \
  --m-metadata-column body-site \
  --p-pairwise \
  --o-visualization unweighted-unifrac-body-site-group-significance.qzv
qiime diversity beta-group-significance \
  --i-distance-matrix diversity-core-metrics-phylogenetic/unweighted_unifrac_distance_matrix.qza \
  --m-metadata-file sample-metadata.tsv \
  --m-metadata-column subject \
  --p-pairwise \
  --o-visualization unweighted-unifrac-subject-group-significance.qzv


Saved Visualization to: unweighted-unifrac-body-site-group-significance.qzv
Saved Visualization to: unweighted-unifrac-subject-group-significance.qzv


> **Question：** subject 與 body-site 對群落組成差異是否具統計顯著性？哪些 body-site 兩兩配對間差異顯著？


### Emperor PCoA（加入時間軸 `days-since-experiment-start`）

In [22]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime emperor plot \
  --i-pcoa diversity-core-metrics-phylogenetic/unweighted_unifrac_pcoa_results.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-custom-axes days-since-experiment-start \
  --o-visualization unweighted-unifrac-emperor-days-since-experiment-start.qzv
qiime emperor plot \
  --i-pcoa diversity-core-metrics-phylogenetic/bray_curtis_pcoa_results.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-custom-axes days-since-experiment-start \
  --o-visualization bray-curtis-emperor-days-since-experiment-start.qzv


Saved Visualization to: unweighted-unifrac-emperor-days-since-experiment-start.qzv
Saved Visualization to: bray-curtis-emperor-days-since-experiment-start.qzv


> **Question：** Emperor 圖是否支持前面 beta diversity 分析的結論？unweighted UniFrac 與 Bray-Curtis 的 PCoA 圖有何差異？


## 8. Alpha rarefaction plotting

檢視 alpha diversity 隨定序深度（sampling depth）變化的曲線，判斷是否已充分定序。
`--p-max-depth` 建議參考 `table.qzv` 的序列數中位數來決定，這裡沿用官方範例值 `4000`。


In [23]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime diversity alpha-rarefaction \
  --i-table table.qza \
  --i-phylogeny rooted-tree.qza \
  --p-max-depth 4000 \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization alpha-rarefaction.qzv


Saved Visualization to: alpha-rarefaction.qzv


> **Question：** 依 body-site 分組看 observed_features 的 rarefaction 曲線，哪些 body-site 曲線有明顯 level off？
> right palm 的曲線在 ~40 附近似乎持平後又跳到 ~140，這可能是什麼原因？（提示：同時看上下兩張圖）


## 9. Taxonomic analysis

> **Warning：** 這裡使用的分類器是用過時的 Greengenes 13_8 訓練的「suboptimal」分類器，
> 只是因為訓練資料小、跑得快，方便教學展示。實際分析請依自己的資料選用合適的（建議使用
> environment-weighted）預訓練分類器。

### 9.1 訓練 16S rRNA 分類器


In [24]:
%%bash
cd qiime2-moving-pictures-tutorial
wget -O 'reference-sequences.qza' \
  'https://moving-pictures-tutorial.readthedocs.io/en/2026.7/data/moving-pictures/reference-sequences.qza'
wget -O 'reference-taxonomy.qza' \
  'https://moving-pictures-tutorial.readthedocs.io/en/2026.7/data/moving-pictures/reference-taxonomy.qza'


--2026-09-04 16:44:16--  https://moving-pictures-tutorial.readthedocs.io/en/2026.7/data/moving-pictures/reference-sequences.qza
Resolving moving-pictures-tutorial.readthedocs.io (moving-pictures-tutorial.readthedocs.io)... 104.16.254.120, 104.16.253.120, 2606:4700::6810:fd78, ...
Connecting to moving-pictures-tutorial.readthedocs.io (moving-pictures-tutorial.readthedocs.io)|104.16.254.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1743450 (1.7M) [application/octet-stream]
Saving to: ‘reference-sequences.qza’

     0K .......... .......... .......... .......... ..........  2% 9.38M 0s
    50K .......... .......... .......... .......... ..........  5%  248M 0s
   100K .......... .......... .......... .......... ..........  8%  249M 0s
   150K .......... .......... .......... .......... .......... 11% 46.3M 0s
   200K .......... .......... .......... .......... .......... 14% 27.1M 0s
   250K .......... .......... .......... .......... .......... 17%  359M 0

In [25]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime feature-classifier fit-classifier-naive-bayes \
  --i-reference-reads reference-sequences.qza \
  --i-reference-taxonomy reference-taxonomy.qza \
  --o-classifier suboptimal-16S-rRNA-classifier.qza


Saved TaxonomicClassifier to: suboptimal-16S-rRNA-classifier.qza


### 9.2 套用分類器

In [26]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime feature-classifier classify-sklearn \
  --i-classifier suboptimal-16S-rRNA-classifier.qza \
  --i-reads rep-seqs.qza \
  --o-classification taxonomy.qza
qiime metadata tabulate \
  --m-input-file taxonomy.qza \
  --o-visualization taxonomy.qzv


Saved FeatureData[Taxonomy] to: taxonomy.qza
Saved Visualization to: taxonomy.qzv


> **Question：** 用 `rep-seqs.qzv` 對照 `taxonomy.qzv`，比較幾個 feature 的分類結果與 BLAST 最佳解的分類是否一致？
> 若不同，是在哪個分類階層（種、屬、科…）開始出現差異？


In [28]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime taxa barplot \
  --i-table table.qza \
  --i-taxonomy taxonomy.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization taxa-bar-plots.qzv


Saved Visualization to: taxa-bar-plots.qzv


> **Question：** 用 Level 2（phylum）檢視，依 body-site → subject → days-since-experiment-start 排序，
> 各 body-site 主要的 phylum 為何？兩位受試者在第 0 天與後續時間點之間是否有一致的變化趨勢？


## 10. Differential abundance testing with ANCOM-BC

先篩選出腸道（gut）樣本，再用 ANCOM-BC 檢定 subject 間差異表現的 feature / genus。


In [29]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime feature-table filter-samples \
  --i-table table.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-where '[body-site]="gut"' \
  --o-filtered-table gut-table.qza


Saved FeatureTable[Frequency] to: gut-table.qza


### 10.1 ASV 層級 ANCOM-BC

In [30]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime composition ancombc \
  --i-table gut-table.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-formula subject \
  --o-differentials ancombc-subject.qza
qiime composition da-barplot \
  --i-data ancombc-subject.qza \
  --o-visualization da-barplot-subject.qzv


Saved FeatureData[DifferentialAbundance] to: ancombc-subject.qza
Saved Visualization to: da-barplot-subject.qzv


> **Question：** 哪個 ASV 相對於參考組（預設 subject-1）最為 enriched？哪個最為 depleted？
> 如果把參考組換成 subject-2，你預期結果會如何變化？


### 10.2 屬（genus，Level 6）層級 ANCOM-BC

In [31]:
%%bash
cd qiime2-moving-pictures-tutorial
qiime taxa collapse \
  --i-table gut-table.qza \
  --i-taxonomy taxonomy.qza \
  --p-level 6 \
  --o-collapsed-table gut-table-l6.qza
qiime composition ancombc \
  --i-table gut-table-l6.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-formula subject \
  --o-differentials l6-ancombc-subject.qza
qiime composition da-barplot \
  --i-data l6-ancombc-subject.qza \
  --o-visualization l6-da-barplot-subject.qzv


Saved FeatureTable[Frequency] to: gut-table-l6.qza
Saved FeatureData[DifferentialAbundance] to: l6-ancombc-subject.qza
Saved Visualization to: l6-da-barplot-subject.qzv


> **Question：** 哪個屬（genus）最 enriched？哪個最 depleted？`da-barplot-subject.qzv`（ASV 層級）與
> `l6-da-barplot-subject.qzv`（genus 層級）相比，哪一個顯著差異的 feature 較多？為什麼會有這種差異？

---

## 分析完成後

所有 `.qza` / `.qzv` 檔案都可以拖曳到 [QIIME 2 View](https://view.qiime2.org) 檢視，
或用 `qiime tools view <file>.qzv` 在有圖形介面的環境中直接開啟。
